# Welcome to Part 2 of the Workshop!

We will now look at how do we *transform* messy json files into nice dataframes using python Pandas!

In [133]:
#run this cell if you need to install pandas
#%pip install pandas

In [134]:
import pandas as pd
from pandas import json_normalize
import json
from datetime import datetime

# How to Read JSON

## How to Read JSON: Key Concepts

### The Anatomy of a JSON Object
A JSON object is built from three primary structural components.

A. The Object `{ }`
   * Definition: The "Main Vault." Represented by curly braces.
   * Function: It acts as a container for data, holding multiple key-value pairs separated by commas.

B. The Key (The Label)
* Requirement: Must always be a string wrapped in double quotes (e.g., "firstName").
* Function: It acts as the unique identifier for a piece of data.

C. The Value (The Loot)
* Supported Types:
   * String: "John" (Must be in double quotes)
   * Number: 30 (No quotes)
   * Boolean: true / false (Lowercase)
   * Null: null (Represents empty/unknown data)
   * Nested Object: A safe within a safe!

2. Lists and Sequences: The Array `[ ]`
* Structure: Represented by square brackets.
* Function: An ordered list of values or objects.
* Example: ["History", "Math", "Science"]

# Loading & Reading Messy JSON Data

If you refer to messy_json.json or scraped_data.json, you will realise that the data is all in 1 line. This is actually common especially when you scrape them from web pages. So here are some tips and links to help with json validating and formatting.

**JSON Validator**
- online JSON validator: https://jsonutils.org/json-validator.html
- to ensure that the data you have adheres strictly to the JSON syntax
- validate before even using the json data!

**JSON Formatter**
- online JSON formatter: https://jsonutils.org/json-formatter.html
- to beautify your JSON in a more readable format

In [ ]:
# load json
with open('messy_json.json') as file:
    data = json.load(file)

print("Original JSON:")
print(json.dumps(data, indent=4)) # prints the json in a readable format

Original JSON:
{
    "id": "001",
    "name": "Alice",
    "created_at": "2023-10-16T15:47:00Z",
    "age": null,
    "contacts": [
        {
            "type": "email",
            "value": "alice@example.com"
        },
        {
            "type": "phone",
            "value": null
        }
    ],
    "address": "123 Main St",
    "preferences": {
        "newsletter": true,
        "sms": "yes"
    },
    "orders": [
        {
            "orderId": 1001,
            "timestamp": 1697468820,
            "items": "apple, banana",
            "total": "15.50"
        },
        {
            "orderId": "1002",
            "timestamp": 1712988368,
            "items": [
                {
                    "name": "orange",
                    "qty": 2
                },
                {
                    "name": "grape",
                    "qty": "3"
                }
            ],
            "total": 20.75
        }
    ]
}


### Multi Cardinality

This refers to the "one-to-many" relationships within your data. In our JSON:
* One User (Alice) has Many contacts
* One User has Many orders
* One Order has Many items

## Accessing items

In [136]:
# let's say we want to access the name in the dictionary
first_person_name = data['name']
print("\nName of the person: ", first_person_name)


Name of the person:  Alice


In [137]:
# now, lets say we want the access to Alice's email contact information
alice_email = None

'''
# we will need to loop through the contacts list to find the email contact for Alice
# * loops are necessary when the field contains multiple items (dictionaries) in a list,
# but they don’t have unique keys like "email" or "phone" as part of their parent object
'''

for contact in data['contacts']:
	if contact['type'] == "email":
		alice_email = contact['value']
		break

print("\nAlice's email:", alice_email)


Alice's email: alice@example.com


## Why this JSON is “messy”:

A key question when looking at messy JSON is: **"What’s wrong with this JSON structure, and why would this be a problem in data analysis?"**

1. Inconsistent data types
* "total" is sometimes a string, sometimes a number.
* "qty" is sometimes a number, sometimes a string.

2. Mixed structures
* "items" is a comma-separated string in one order, but an array of objects in another.
* "preferences.sms" is "yes" instead of a boolean → boolean is more appropriate for binary fields.

3. Irregular formatting
* Some fields are objects, others are strings, and some are arrays for the same logical data.
* date-time is not standardised and not human-friendly.

4. `Null` values
* Ambiguity: A `null` could mean the data was never collected, it doesn't exist, or there was a system error.
* Math Breaking: You cannot perform mathematical operations (like mean or sum) on a column that contains `None` or `NaN` without handling them first.
* Inconsistency: In your JSON, age is `null`. If you try to convert it to an integer, Python will throw an error.

# Cleaning JSON

## Common Functions in Pandas to clean:
* **Filling missing values:**
    * `df.fillna()` method: replacing missing values with a “default” value
    * `df.dropna()` method: removing any rows or columns containing missing values
* **Removing duplicated instances:**
    * `df.drop_duplicates()` method: automatically removing duplicate entries (rows) in a dataset => which allows the removal of extra instances when either a specific attribute value or the entire instance values are duplicated to another entry
* **Manipulating strings:**
    * `df['column'].str.lower()`: use if there is a mix of lowercase, sentencecase, and uppercase values for an 'column' attribute and we want them all to be lowercase
    * `df['column'].str.strip()`: For removing accidentally introduced leading and trailing whitespaces
* **Manipulating date and time:**
    * `pd.to_datetime(df['column'])` converts string columns containing date-time information (e.g. in the dd/mm/yyyy format) into Python datetime objects, thereby easing their further manipulation
* **Column renaming:**
    * `df.rename(columns={old_name: new_name})`
    *  useful when there are multiple datasets seggregated by city, region, project, etc., and we want to add prefixes or suffixes to all or some of their columns for easing their identification

In [138]:
# Handle nulls at the dictionary level before Pandas
# NOTE: We only replace null for specific known fields to avoid masking data in nested structures
def clean_nulls(d):
    if isinstance(d, dict):
        return {k: (v if v is not None else "Unknown") for k, v in d.items()}
    return d

# Clean the root level only (fixes 'age' -> 'Unknown')
# Note: this does NOT recurse into nested lists like contacts or orders,
# so nulls within those structures are handled separately below
data = clean_nulls(data)
print(data)

# to format to json for readability
print(json.dumps(data, indent=4))

{'id': '001', 'name': 'Alice', 'created_at': '2023-10-16T15:47:00Z', 'age': 'Unknown', 'contacts': [{'type': 'email', 'value': 'alice@example.com'}, {'type': 'phone', 'value': None}], 'address': '123 Main St', 'preferences': {'newsletter': True, 'sms': 'yes'}, 'orders': [{'orderId': 1001, 'timestamp': 1697468820, 'items': 'apple, banana', 'total': '15.50'}, {'orderId': '1002', 'timestamp': 1712988368, 'items': [{'name': 'orange', 'qty': 2}, {'name': 'grape', 'qty': '3'}], 'total': 20.75}]}
{
    "id": "001",
    "name": "Alice",
    "created_at": "2023-10-16T15:47:00Z",
    "age": "Unknown",
    "contacts": [
        {
            "type": "email",
            "value": "alice@example.com"
        },
        {
            "type": "phone",
            "value": null
        }
    ],
    "address": "123 Main St",
    "preferences": {
        "newsletter": true,
        "sms": "yes"
    },
    "orders": [
        {
            "orderId": 1001,
            "timestamp": 1697468820,
           

In [139]:
# Clean the contacts: replace null phone value with 'Unknown'
for contact in data["contacts"]:
    if contact["type"] == "phone":
        contact["value"] = contact["value"] if contact["value"] is not None else "Unknown"

print("Contacts after cleaning:", data["contacts"])

Contacts after cleaning: [{'type': 'email', 'value': 'alice@example.com'}, {'type': 'phone', 'value': 'Unknown'}]


In [140]:
# convert 'id' field to integer (assuming that the zeros are not significant and that 'id' should be a number)
data["id"] = int(data["id"])

print("ID field:", data["id"])
print("ID type:", type(data["id"]))
print(json.dumps(data, indent=4))

ID field: 1
ID type: <class 'int'>
{
    "id": 1,
    "name": "Alice",
    "created_at": "2023-10-16T15:47:00Z",
    "age": "Unknown",
    "contacts": [
        {
            "type": "email",
            "value": "alice@example.com"
        },
        {
            "type": "phone",
            "value": "Unknown"
        }
    ],
    "address": "123 Main St",
    "preferences": {
        "newsletter": true,
        "sms": "yes"
    },
    "orders": [
        {
            "orderId": 1001,
            "timestamp": 1697468820,
            "items": "apple, banana",
            "total": "15.50"
        },
        {
            "orderId": "1002",
            "timestamp": 1712988368,
            "items": [
                {
                    "name": "orange",
                    "qty": 2
                },
                {
                    "name": "grape",
                    "qty": "3"
                }
            ],
            "total": 20.75
        }
    ]
}


In [141]:
# Clean preferences (convert "yes" in sms to boolean True)
data["preferences"]["sms"] = data["preferences"]["sms"] == "yes"

print("Preferences after cleaning:", data["preferences"])

Preferences after cleaning: {'newsletter': True, 'sms': True}


In [142]:
# Clean orders
for order in data["orders"]:
    # Convert orderId to integer
    order["orderId"] = int(order["orderId"])

    # Convert total to a float
    order["total"] = float(order["total"])

    # Handle items
    if isinstance(order["items"], str):
        # If items are in a string format, split into a list of dictionaries
        items = order["items"].split(", ")
        order["items"] = [{"name": item, "qty": 1} for item in items]
    else:
        # Ensure qty is an integer in the item list
        for item in order["items"]:
            item["qty"] = int(item["qty"])

In [143]:
# display what has been cleaned so far
cleaned_data_json = json.dumps(data, indent=4)
print(cleaned_data_json)

{
    "id": 1,
    "name": "Alice",
    "created_at": "2023-10-16T15:47:00Z",
    "age": "Unknown",
    "contacts": [
        {
            "type": "email",
            "value": "alice@example.com"
        },
        {
            "type": "phone",
            "value": "Unknown"
        }
    ],
    "address": "123 Main St",
    "preferences": {
        "newsletter": true,
        "sms": true
    },
    "orders": [
        {
            "orderId": 1001,
            "timestamp": 1697468820,
            "items": [
                {
                    "name": "apple",
                    "qty": 1
                },
                {
                    "name": "banana",
                    "qty": 1
                }
            ],
            "total": 15.5
        },
        {
            "orderId": 1002,
            "timestamp": 1712988368,
            "items": [
                {
                    "name": "orange",
                    "qty": 2
                },
                {
    

## JSON Timestamps

### What Are Timestamps?

Timestamps are representations of dates and times in a database or JSON, often stored in a standard or machine-readable format. However, these formats are not always human-readable and may require conversion.

### Formats of Timestamps:
1. **ISO 8601 format**: Common in JSON
    * Example: "2023-10-16T15:47:00Z"
    * Includes both date and time, as well as time zone information.
2. **Unix Timestamp**: Common in APIs
    * Example: 1697468820
    * Stores the number of seconds (or sometimes milliseconds) since 1970-01-01 00:00:00 (UTC).
3. **Other formats**:
    * Sometimes timestamps are in non-standard formats (e.g: "20231016").

### Why Convert Timestamps?
* Tools like Pandas or SQL need consistent and proper datetime types for data processing.
* Converting timestamps makes it easier to:
    * Parse timestamps for analysis (e.g., sorting by date).
    * Perform time-related operations (e.g., filtering or grouping data by year, quarter, or hour).
Make timestamps human-readable.

In [144]:
# Convert order timestamps (Unix seconds -> datetime)
for order in data["orders"]:
    order["timestamp"] = pd.to_datetime(order["timestamp"], unit='s')

# Convert root-level "created_at" (ISO 8601 -> datetime)
data["created_at"] = pd.to_datetime(data["created_at"])

print("Orders after timestamp conversion:")
print(data["orders"])
print("\nRoot-level created_at:")
print(data["created_at"])

Orders after timestamp conversion:
[{'orderId': 1001, 'timestamp': Timestamp('2023-10-16 15:07:00'), 'items': [{'name': 'apple', 'qty': 1}, {'name': 'banana', 'qty': 1}], 'total': 15.5}, {'orderId': 1002, 'timestamp': Timestamp('2024-04-13 06:06:08'), 'items': [{'name': 'orange', 'qty': 2}, {'name': 'grape', 'qty': 3}], 'total': 20.75}]

Root-level created_at:
2023-10-16 15:47:00+00:00


## JSON Flattening

Using **`pd.json_normalize`**

Designed to transform semi-structured JSON data, such as nested dictionaries or lists, into a flat table
This is particularly useful when handling JSON-like data structures that contain deeply nested fields

**Important Arguments:**

1. data:
    * unserialized json objects
    * accepts the JSON-like data structure that you want to normalize/ flatten
2. record_path (`str` or `list of str`):
    * path in each object to list of records, where these records will be expanded into **rows** in the resulting DataFrame
3. meta (`list of str` or `list of list of str`):
    * fields to use as metadata for each record in resulting table;
    * specifies fields (or paths to fields) that should be preserved as metadata and attached to each row in the resulting DataFrame

more info on https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.json_normalize.html


In [145]:
# Example 1: Extracting items from orders and creating a DataFrame
# Convert orders to a Pandas df for further operations

orders_df = pd.json_normalize(
    data["orders"],
    record_path="items",                            # expand the items list into rows
    #meta=["id", "name", ["orders", "orderId"]]     # uncomment to see the difference when we include more metadata fields
)

print(orders_df)

     name  qty
0   apple    1
1  banana    1
2  orange    2
3   grape    3


In [146]:
# Example 2: Flattening orders with custom column names
orders_flat_df = pd.json_normalize(
    data['orders'],
    record_path='items',
    meta=['orderId'],
)
# Add customer name from root to all rows
orders_flat_df['customer_name'] = data['name']

# Rename columns
orders_flat_df = orders_flat_df.rename(columns={
    'name': 'item_name',
    'qty': 'quantity',
    'orderId': 'order_id'
})

print(orders_flat_df)

  item_name  quantity order_id customer_name
0     apple         1     1001         Alice
1    banana         1     1001         Alice
2    orange         2     1002         Alice
3     grape         3     1002         Alice


In [ ]:
# export to csv
orders_df.to_csv("cleaned_orders.csv", index=False)
orders_flat_df.to_csv("flattened_orders.csv", index=False)

For more info on Pandas documentation, you can refer to this cheatsheet: https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf

# Looking at the JSON scrapped previously...

Let's now start to clean & flatten the json that we derived from the web page!

In [ ]:
# 1. Load the JSON data
with open('scraped_data.json', 'r') as f:
    data = json.load(f)

In [ ]:
# 2. Basic validation function to check if JSON is well-formed
def validate_json_file(filepath):
    try:
        with open(filepath, 'r') as f:
            data = json.load(f)
        print("✅ JSON is valid and well-formatted!")
        # Optionally, pretty-print the JSON
        print(json.dumps(data, indent=2)[:1000])  # Print first 1000 chars for preview
        return True
    except json.JSONDecodeError as e:
        print(f"❌ JSON is invalid: {e}")
        return False

validate_json_file('messy_json.json')

✅ JSON is valid and well-formatted!
{
  "id": "001",
  "name": "Alice",
  "created_at": "2023-10-16T15:47:00Z",
  "age": null,
  "contacts": [
    {
      "type": "email",
      "value": "alice@example.com"
    },
    {
      "type": "phone",
      "value": null
    }
  ],
  "address": "123 Main St",
  "preferences": {
    "newsletter": true,
    "sms": "yes"
  },
  "orders": [
    {
      "orderId": 1001,
      "timestamp": 1697468820,
      "items": "apple, banana",
      "total": "15.50"
    },
    {
      "orderId": "1002",
      "timestamp": 1712988368,
      "items": [
        {
          "name": "orange",
          "qty": 2
        },
        {
          "name": "grape",
          "qty": "3"
        }
      ],
      "total": 20.75
    }
  ]
}


True

In [150]:
# 3. Flatten the data using json_normalize

'''
# Nested dictionary → flattened automatically into columns
"Metadata": {
    "Capital": "Andorra la Vella",
    "Population": "84000"
}
# Result: Metadata_Capital, Metadata_Population

# Nested list → needs record_path to expand into rows
"orders": [
    {"orderId": 1001, ...},
    {"orderId": 1002, ...}
]
# Result: needs record_path="orders" to handle
'''

flat_df = json_normalize(data, sep='_')
display(flat_df)

,Country,Metadata_Capital,Metadata_Population,Metadata_Area (km^2)
0,Andorra,Andorra la Vella,84000,468.0
1,United Arab Emirates,Abu Dhabi,4975593,82880.0
2,Afghanistan,Kabul,29121286,647500.0
3,Antigua and Barbuda,St. John's,86754,443.0
4,Anguilla,The Valley,13254,102.0
...,...,...,...,...
245,Yemen,Sanaa,23495361,527970.0
246,Mayotte,Mamoudzou,159042,374.0
247,South Africa,Pretoria,49000000,1219912.0
248,Zambia,Lusaka,13460305,752614.0


In [151]:
# 4. Check if its cleaned after flattening
def check_cleaning_df(df):
    print("--- DataFrame Cleaning Check ---")
    print("Shape:", df.shape)
    print("\nData types:\n", df.dtypes)
    print("\nNull counts:\n", df.isnull().sum())
    print("\nSample:\n", df.head())

check_cleaning_df(flat_df)

--- DataFrame Cleaning Check ---
Shape: (250, 4)

Data types:
 Country                 object
Metadata_Capital        object
Metadata_Population     object
Metadata_Area (km^2)    object
dtype: object

Null counts:
 Country                 0
Metadata_Capital        0
Metadata_Population     0
Metadata_Area (km^2)    0
dtype: int64

Sample:
                 Country  Metadata_Capital Metadata_Population  \
0               Andorra  Andorra la Vella               84000   
1  United Arab Emirates         Abu Dhabi             4975593   
2           Afghanistan             Kabul            29121286   
3   Antigua and Barbuda        St. John's               86754   
4              Anguilla        The Valley               13254   

  Metadata_Area (km^2)  
0                468.0  
1              82880.0  
2             647500.0  
3                443.0  
4                102.0  


In [ ]:
# 5. Export the flattened df to csv
flat_df.to_csv('flat_data.csv', index=False)